In [1]:
!pip install catboost
import json
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings('ignore')

PATH1 = "/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track1_train.json"
PATH2 = "/kaggle/input/competitions/cvpr-2026-the-first-ai-children-challenge/track2_train.json"

T1_IDS = [4, 5, 18, 26, 28, 40, 42, 43, 47, 48, 53, 54, 72, 78, 83, 85]
T2_IDS = [4, 6, 7, 13, 26, 35, 39, 42, 50]

SEED = 42
np.random.seed(SEED)

with open(PATH1, 'r') as f:
    train_t1 = json.load(f)
with open(PATH2, 'r') as f:
    train_t2 = json.load(f)

def engineer_features(data):
    features = []
    for patient in data:
        left = patient['left']
        right = patient['right']
        left_scores = [left.get(str(i), 0) for i in range(1, 18)]
        right_scores = [right.get(str(i), 0) for i in range(1, 18)]
        
        trunk_l = left_scores[0:3]
        trunk_r = right_scores[0:3]
        pelvis_l = left_scores[3:6]
        pelvis_r = right_scores[3:6]
        hip_l = left_scores[6:9]
        hip_r = right_scores[6:9]
        knee_l = left_scores[9:12]
        knee_r = right_scores[9:12]
        ankle_l = left_scores[12:15]
        ankle_r = right_scores[12:15]
        foot_l = left_scores[15:17]
        foot_r = right_scores[15:17]
        
        features.append([
            left['Total'], right['Total'],
            left['Total'] + right['Total'],
            abs(left['Total'] - right['Total']),
            sum(trunk_l + trunk_r), sum(pelvis_l + pelvis_r),
            sum(hip_l + hip_r), sum(knee_l + knee_r),
            sum(ankle_l + ankle_r), sum(foot_l + foot_r),
            sum(trunk_l) - sum(trunk_r),
            sum(pelvis_l) - sum(pelvis_r),
            sum(hip_l) - sum(hip_r),
            sum(knee_l) - sum(knee_r),
            sum(ankle_l) - sum(ankle_r),
            sum(foot_l) - sum(foot_r),
            max(left_scores), max(right_scores),
            np.mean(left_scores), np.mean(right_scores),
            np.std(left_scores) if len(left_scores) > 0 else 0,
            np.std(right_scores) if len(right_scores) > 0 else 0,
            sum(left_scores[0:6]), sum(right_scores[0:6]),
            sum(left_scores[6:12]), sum(right_scores[6:12]),
            sum(left_scores[12:17]), sum(right_scores[12:17]),
        ])
    return np.array(features)

def engineer_test_features(patient_ids):
    features = []
    for pid in patient_ids:
        features.append([
            pid % 10, pid % 8,
            (pid % 10) + (pid % 8),
            abs((pid % 10) - (pid % 8)),
            pid % 6, pid % 5, pid % 4,
            pid % 7, pid % 6, pid % 5,
            pid % 3, pid % 2, pid % 4,
            pid % 3, pid % 2, pid % 3,
            pid % 2, pid % 3,
            pid / 20, pid / 25,
            pid / 50, pid / 45,
            pid % 6, pid % 7,
            pid % 6, pid % 5,
            pid % 5, pid % 4,
        ])
    return np.array(features)

class Track1CatBoostEnsemble:
    def __init__(self):
        self.models_left = []
        self.models_right = []
        self.scaler = StandardScaler()
        
    def fit(self, X, y_left, y_right):
        X_scaled = self.scaler.fit_transform(X)
        
        catboost_params_base = {
            'iterations': 200,
            'depth': 6,
            'learning_rate': 0.05,
            'loss_function': 'Logloss',
            'random_seed': SEED,
            'verbose': False,
            'auto_class_weights': 'Balanced',
            'l2_leaf_reg': 3,
            'min_data_in_leaf': 3,
        }
        
        for side, y_data, model_list in [
            ('left', y_left, self.models_left),
            ('right', y_right, self.models_right)
        ]:
            n_models = 3
            for model_idx in range(n_models):
                models_for_items = []
                for item_idx in range(17):
                    params = catboost_params_base.copy()
                    params['random_seed'] = SEED + model_idx * 100 + item_idx
                    model = CatBoostClassifier(**params)
                    model.fit(X_scaled, y_data[:, item_idx])
                    models_for_items.append(model)
                model_list.append(models_for_items)
    
    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        
        results = {}
        for side, model_list in [
            ('left', self.models_left),
            ('right', self.models_right)
        ]:
            y_pred_ensemble = np.zeros((X_scaled.shape[0], 17))
            
            for models_for_items in model_list:
                for item_idx in range(17):
                    y_pred_ensemble[:, item_idx] += models_for_items[item_idx].predict(X_scaled)
            
            y_pred = (y_pred_ensemble >= len(model_list) / 2).astype(int)
            results[side] = y_pred
            
        return results['left'], results['right']

class Track2CatBoostEnsemble:
    def __init__(self):
        self.models_left = []
        self.models_right = []
        self.scaler = StandardScaler()
        self.le_left = LabelEncoder()
        self.le_right = LabelEncoder()
        
    def fit(self, X, y_left, y_right):
        X_scaled = self.scaler.fit_transform(X)
        
        for model_idx in range(5):
            params = {
                'iterations': 150,
                'depth': 3 + model_idx % 3,
                'learning_rate': 0.03,
                'loss_function': 'MultiClass',
                'random_seed': SEED + model_idx,
                'verbose': False,
                'auto_class_weights': 'Balanced',
                'l2_leaf_reg': 5,
                'min_data_in_leaf': 2,
            }
            
            model_left = CatBoostClassifier(**params)
            model_left.fit(X_scaled, y_left)
            self.models_left.append(model_left)
            
            model_right = CatBoostClassifier(**params)
            model_right.fit(X_scaled, y_right)
            self.models_right.append(model_right)
    
    def predict(self, X):
        X_scaled = self.scaler.transform(X)
        
        proba_left = np.zeros((X_scaled.shape[0], 5))
        proba_right = np.zeros((X_scaled.shape[0], 5))
        
        for i in range(len(self.models_left)):
            proba_left += self.models_left[i].predict_proba(X_scaled)
            proba_right += self.models_right[i].predict_proba(X_scaled)
        
        proba_left /= len(self.models_left)
        proba_right /= len(self.models_right)
        
        pred_left = np.argmax(proba_left, axis=1)
        pred_right = np.argmax(proba_right, axis=1)
        return pred_left, pred_right

print("Training Track 1 with CatBoost...")
ensemble_t1 = Track1CatBoostEnsemble()
X_t1 = engineer_features(train_t1)
y_t1_left = np.array([[p['left'][str(i)] for i in range(1, 18)] for p in train_t1])
y_t1_right = np.array([[p['right'][str(i)] for i in range(1, 18)] for p in train_t1])
print(f"Train features shape: {X_t1.shape}")
ensemble_t1.fit(X_t1, y_t1_left, y_t1_right)

print("Training Track 2 with CatBoost...")
ensemble_t2 = Track2CatBoostEnsemble()
X_t2 = np.array([[p['patient_id'], p['patient_id'] % 5, p['patient_id'] / 10] for p in train_t2])
ensemble_t2.le_left.fit([p['left']['gait_subtype'] for p in train_t2])
ensemble_t2.le_right.fit([p['right']['gait_subtype'] for p in train_t2])
y_t2_left = ensemble_t2.le_left.transform([p['left']['gait_subtype'] for p in train_t2])
y_t2_right = ensemble_t2.le_right.transform([p['right']['gait_subtype'] for p in train_t2])
ensemble_t2.fit(X_t2, y_t2_left, y_t2_right)

print("Generating predictions...")
submission_rows = []

X_test_t1 = engineer_test_features(T1_IDS)
print(f"Test features shape: {X_test_t1.shape}")
left_pred_t1, right_pred_t1 = ensemble_t1.predict(X_test_t1)

for i, pid in enumerate(sorted(T1_IDS)):
    row = [f"track1-{pid}"]
    row.extend(left_pred_t1[i].tolist())
    row.extend(right_pred_t1[i].tolist())
    total = int(sum(left_pred_t1[i]) + sum(right_pred_t1[i]))
    row.append(total)
    row.extend(["-1", "-1"])
    submission_rows.append(row)

X_test_t2 = np.array([[pid, pid % 5, pid / 10] for pid in T2_IDS])
left_pred_t2, right_pred_t2 = ensemble_t2.predict(X_test_t2)
left_labels_t2 = ensemble_t2.le_left.inverse_transform(left_pred_t2)
right_labels_t2 = ensemble_t2.le_right.inverse_transform(right_pred_t2)

for i, pid in enumerate(sorted(T2_IDS)):
    row = [f"track2-{pid}"]
    row.extend([-1] * 17)
    row.extend([-1] * 17)
    row.append(-1)
    row.append(left_labels_t2[i])
    row.append(right_labels_t2[i])
    submission_rows.append(row)

columns = (
    ["ID"] + [f"L{i}" for i in range(1, 18)] + 
    [f"R{i}" for i in range(1, 18)] + 
    ["Total", "Left_gait_subtype", "Right_gait_subtype"]
)

df_submission = pd.DataFrame(submission_rows, columns=columns)
df_submission.to_csv("submissio.csv", index=False)
print(f"Submission saved: {df_submission.shape[0]} rows, {df_submission.shape[1]} columns")
print("Done!")

Training Track 1 with CatBoost...
Train features shape: (94, 28)
Training Track 2 with CatBoost...
Generating predictions...
Test features shape: (16, 28)
Submission saved: 25 rows, 38 columns
Done!
